# 03 Carga Politecnica Disponible

## Propósito

Esta fuente contiene **actividades de carga politécnica y responsabilidades institucionales.**

Forma parte de la base de conocimiento para perfiles multidimensionales del personal. No se usa aquí para clasificar, predecir ni tomar decisiones sobre personas.

## Reglas de seguridad y conservación

- No se eliminan filas ni columnas de origen.
- No se imputan valores faltantes.
- Los identificadores se mantienen como texto para proteger ceros iniciales y códigos.
- La fuente original no se modifica; el resultado se guarda como un archivo nuevo.
- Una fecha o cantidad sólo se convierte si puede interpretarse; la versión original siempre permanece disponible.

## Convención de las variables nuevas

| Sufijo | Significado | Ejemplo |
| --- | --- | --- |
| __FECHA | Fecha interpretada | FECHAINICIO__FECHA |
| __FECHA_VALIDA | Indica si la fecha fue interpretable | FECHAINICIO__FECHA_VALIDA |
| __NUMERICO | Copia numérica de una cantidad | DURACION__NUMERICO |
| __NUMERICO_VALIDO | Indica si el número fue interpretable | DURACION__NUMERICO_VALIDO |

De esta forma, un valor problemático no se pierde ni se modifica silenciosamente.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

# Buscar la raíz del proyecto para que funcione desde VS Code o Jupyter.
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "data" / "cruda").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "notebooks" / "preprocesamiento"))
from _preprocesamiento_comun import process_source


## 1. Ejemplos reales de la fuente

Se carga una muestra sin transformaciones. Revise nombres de columnas, códigos y formatos antes de usar cualquier variable en un análisis.

In [ ]:
fuente = ROOT / "data" / "cruda" / "cargapolitecnicadisponible.csv"
salida = ROOT / "data" / "processed" / "03_carga_politecnica_disponible.parquet"

# Lectura exploratoria: todos los campos se conservan como texto.
datos_crudos = pd.read_csv(fuente, dtype="string", keep_default_na=False, encoding="utf-8-sig")
print(f"Filas: {len(datos_crudos):,} | Columnas: {len(datos_crudos.columns):,}")
datos_crudos.head(5)


## 2. Diagnóstico inicial

Esta tabla no altera datos: muestra vacíos y un ejemplo real de cada campo. Úsela para decidir, con expertos institucionales, qué variables son pertinentes.

In [ ]:
perfil = pd.DataFrame({
    "tipo_leido": datos_crudos.dtypes.astype(str),
    "vacios_en_origen": (datos_crudos == "").sum(),
    "ejemplo": [datos_crudos[c].dropna().iloc[0] if len(datos_crudos[c].dropna()) else pd.NA for c in datos_crudos.columns],
})
perfil


## 3. Limpieza y transformación aplicada

El proceso hace únicamente estas operaciones:

1. Añade FILA_ORIGEN para trazabilidad.
2. Quita espacios de borde y convierte textos vacíos a NA.
3. Para campos de fecha, crea fecha, año, mes e indicador de validez en columnas adicionales.
4. Para cantidades identificables, crea una copia numérica y un indicador de validez.
5. Calcula variables específicas sólo cuando hay columnas suficientes: por ejemplo duración entre fechas, tasa de participación de heteroevaluación o descripciones de códigos de experiencia externa del TXT proporcionado.

No se estandarizan categorías ambiguas ni se eliminan posibles atípicos: esas decisiones requieren validación institucional.

In [ ]:
# Ejecutar el preprocesamiento conservador.
reporte = process_source(fuente, salida, kind="carga_politecnica")
pd.Series(reporte)


## 4. Revisar las variables derivadas

Las siguientes celdas enumeran y muestran ejemplos de las nuevas columnas. Verifique que tengan sentido para esta fuente antes de incorporarlas a la tabla integrada de perfiles.

In [ ]:
datos_procesados = pd.read_parquet(salida)
nuevas = [c for c in datos_procesados.columns if c not in datos_crudos.columns and c != "FILA_ORIGEN"]
print(f"Columnas nuevas: {len(nuevas)}")
pd.DataFrame({"variable_nueva": nuevas})


## 5. Verificación final y entregables

La verificación debe mostrar cero filas eliminadas y True en la conservación de columnas. Además del Parquet se genera un archivo de calidad JSON con nulos por campo, criterios aplicados y trazabilidad.

In [ ]:
verificacion = pd.DataFrame({
    "comprobacion": ["filas de origen", "filas procesadas", "filas eliminadas", "columnas de origen conservadas"],
    "resultado": [len(datos_crudos), len(datos_procesados), len(datos_crudos) - len(datos_procesados), set(datos_crudos.columns).issubset(datos_procesados.columns)],
})
display(verificacion)
display(datos_procesados[nuevas].head(5) if nuevas else pd.DataFrame())
